# Preview the documents prepared for retrieval

**Aim:** See the text table produced from the decisions in notebooks 02–04, and check that every row can be traced to its source.

A **document** here is one piece of text attached to a book: a prepared TOC item or the book's joined Amazon text. A TOC item is often just a heading, not the full chapter. One book can contribute many documents.

[src/bookpath/documents.py](../src/bookpath/documents.py) defines `build_documents(books, toc_features)`, which returns the table in memory. [src/bookpath/pipeline.py](../src/bookpath/pipeline.py) reads the saved inputs, checks the TOC against the raw books and current code, and saves `data/processed/documents.parquet`. Create it from the repository root:

```sh
uv run python -m bookpath.pipeline
```

The command requires the existing raw and TOC files. It protects existing output files; use a different `--output` path for a later experiment. This notebook reads the saved result and rebuilds expected tables in memory for validation. It writes no data files.

| Section | What we learn or validate |
| --- | --- |
| [1. Load the files](#section-1) | Identify the inputs, output, and all 11 output columns. |
| [2. Follow one book](#section-2) | See its TOC documents and the chosen book-level text. |
| [3. Count the results](#section-3) | Reconcile source totals, fallback use, omitted empty text, and book coverage. |
| [4. Validate the saved table](#section-4) | Check identifiers, text, both preparation steps, and unchanged files. |

These are ordinary text strings. No tokenizer, splitting into model-sized passages, embeddings, or ranking is implemented yet. Findings describe the saved local snapshot.

<a id="section-1"></a>

## 1. What files and columns are we using?

**Objective:** Load the original books, their prepared TOC rows, and the new document table. Record each file's checksum to check later that it remains unchanged.

The three tables use different units: one raw row per book, one prepared TOC row per original TOC item, and one document row per selected nonblank text.

In [1]:
import hashlib
import html
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

from bookpath.documents import build_documents
from bookpath.toc import build_toc_features

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the BookPath repository.")
    project_root = project_root.parent

raw_path = project_root / "data/raw/books.parquet"
toc_path = project_root / "data/processed/toc_features.parquet"
documents_path = project_root / "data/processed/documents.parquet"
if not documents_path.is_file():
    raise FileNotFoundError("Run uv run python -m bookpath.pipeline from the repository root first.")


def file_checksum(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()


checksums_before = {
    path: file_checksum(path) for path in [raw_path, toc_path, documents_path]
}
books = pd.read_parquet(raw_path)
toc_features = pd.read_parquet(toc_path)
documents = pd.read_parquet(documents_path)
print(f"Raw books: {len(books):,} book records.")
print(f"TOC preparation: {len(toc_features):,} original-item rows.")
print(f"Documents: {len(documents):,} rows and {len(documents.columns)} columns.")
print("Document version:", documents["document_version"].unique().tolist())

Raw books: 9,034 book records.
TOC preparation: 141,513 original-item rows.
Documents: 150,496 rows and 11 columns.
Document version: ['documents-v1']


**Where do the 11 columns come from?** The builder creates nine columns; the saving step adds two file checksums. These are the names in `documents`, the table loaded from `documents.parquet`:

| Output column | Meaning and origin |
| --- | --- |
| `document_id` | A new ID: `parent_asin:toc:position` for a TOC item, or `parent_asin:features` / `parent_asin:description` for book-level text. It identifies a source location, not a fixed version of its contents. |
| `parent_asin` | The original book ID, kept as text so leading zeros remain. It repeats when a book has multiple documents. |
| `book_title` | Copied from the raw book's `title`. It is metadata, not appended to `text`. |
| `source` | A new label: `toc`, `features`, or `description`. |
| `raw_position` | The original TOC item's position. Missing for book-level text. |
| `text` | TOC `feature_text`, joined Amazon `features`, or joined `description` when features has no nonblank text. |
| `numeric_only_text` | Copied TOC review flag. Missing for book-level rows because that flag was not assessed there. |
| `repeated_text` | Copied TOC review flag, including every occurrence in a repeated group. Missing for book-level rows. |
| `document_version` | The rule version, `documents-v1`. |
| `raw_source_sha256` | The saving step's checksum of `data/raw/books.parquet`. |
| `toc_source_sha256` | The saving step's checksum of `data/processed/toc_features.parquet`. |

**Conclusion:** The output keeps selected text together with the book, source location, review flags, and input-file identity needed to trace it.

<a id="section-2"></a>

## 2. What did one book contribute?

**Objective:** Follow one book into the output. Leave `example_asin = None` for a repeatable random choice; change `random_seed` for another example, or enter an existing quoted ID. Rerun both cells in this section after changing it. Sections 3–4 always cover all books.

The first table shows up to five TOC documents. The second shows the single book-level document, if present, with a short text preview and an expandable full value. The builder joins each saved list in order, trims the ends of its pieces, and skips blank pieces. It prefers `features`, uses `description` only when features has no text, and creates no book-level document when both are empty.

In [2]:
example_asin = None  # Or a specific ID, such as "0142000280".
random_seed = 42

if example_asin is None:
    example_asin = books.sample(n=1, random_state=random_seed)["parent_asin"].iloc[0]
selected = books.loc[books["parent_asin"].eq(example_asin)]
if selected.empty:
    raise ValueError(f"No saved book has parent_asin = {example_asin!r}.")
selected_book = selected.iloc[0]
book_documents = documents.loc[documents["parent_asin"].eq(example_asin)]
print(f"{selected_book['title']} (parent_asin = {example_asin})")

Carbonell Museum Studies: An Anthology of Contexts (parent_asin = 0631228306)


In [3]:
toc_documents = book_documents.loc[book_documents["source"].eq("toc")]
book_text = book_documents.loc[book_documents["source"].ne("toc")]
print(f"TOC documents: showing up to five of {len(toc_documents)}.")
with pd.option_context("display.max_colwidth", None):
    display(toc_documents[["document_id", "raw_position", "text"]].head(5))

if book_text.empty:
    print("No book-level document: features and description have no nonblank text.")
else:
    print("Book-level document: features was chosen, or description supplied the fallback.")
    preview = book_text[["document_id", "source"]].copy()
    preview["Beginning (up to 200 characters)"] = book_text["text"].str[:200]
    with pd.option_context("display.max_colwidth", None):
        display(preview)
    entry = book_text.iloc[0]
    display(HTML(
        f"<details><summary>Read the complete {html.escape(entry['source'])} text</summary>"
        f"<div style='max-width: 90ch; white-space: pre-wrap'>{html.escape(entry['text'])}</div></details>"
    ))

display(Markdown(
    f"**Conclusion:** Book `{example_asin}` contributes **{len(toc_documents)} TOC documents** "
    f"and **{len(book_text)} book-level document**, giving **{len(book_documents)} documents in total**. "
    "TOC positions refer to the original items, even when an empty item was skipped."
))

TOC documents: showing up to five of 54.


,document_id,raw_position,text
52329,0631228306:toc:1,1,"Introduction: Museum/studies and the ""eccentric space"" of an anthology / Bettina Messias Carbonell"
52330,0631228306:toc:2,2,From The museum age : foreword / Germain Bazin
52331,0631228306:toc:3,3,The museum : its classical etymology and renaissance genealogy / Paula Findlen
52332,0631228306:toc:4,4,The universal survey museum / Carol Duncan and Alan Wallach
52333,0631228306:toc:5,5,Brain of the earth's body : museums and the framing of modernity / Donald Preziosi


Book-level document: features was chosen, or description supplied the fallback.


,document_id,source,Beginning (up to 200 characters)
52383,0631228306:features,features,"Museum Studies: An Anthology of Contexts\nprovides a comprehensive interdisciplinary collection of approaches to museums and their relation to history, culture, philosophy and their adoring or combativ"


**Conclusion:** Book `0631228306` contributes **54 TOC documents** and **1 book-level document**, giving **55 documents in total**. TOC positions refer to the original items, even when an empty item was skipped.

<a id="section-3"></a>

## 3. Do the totals follow our selection rules?

**Objective:** Count documents by source and account for every raw book.

The first table counts document rows and distinct books for each source. **Document counts add up; book counts overlap** because a book can supply both TOC and book-level text. The final row counts each book once.

In [4]:
source_counts = documents.groupby("source").agg(
    Documents=("document_id", "size"),
    Books=("parent_asin", "nunique"),
).reindex(["toc", "features", "description"], fill_value=0)
source_counts.loc["Total (distinct books)"] = [len(documents), documents["parent_asin"].nunique()]
display(source_counts)

skipped_toc = len(toc_features) - int(source_counts.loc["toc", "Documents"])
print(f"TOC reconciliation: {len(toc_features):,} prepared TOC rows minus {skipped_toc:,} "
      f"without nonblank text = {source_counts.loc['toc', 'Documents']:,} TOC documents.")
print(f"Retained TOC flags: {int(documents['numeric_only_text'].sum()):,} numeric-only; "
      f"{int(documents['repeated_text'].sum()):,} repeated. These counts can overlap.")

,Documents,Books
source,,
toc,141495,9034
features,8991,8991
description,10,10
Total (distinct books),150496,9034


TOC reconciliation: 141,513 prepared TOC rows minus 18 without nonblank text = 141,495 TOC documents.
Retained TOC flags: 435 numeric-only; 2,916 repeated. These counts can overlap.


The second table places **every book in exactly one group**. A book with features or description can also have TOC documents. **TOC only** means it has TOC documents but neither book-level text. **No documents** means no source supplied nonblank text. These four groups add up to the total number of raw books.

In [5]:
book_level = documents.loc[documents["source"].ne("toc")]
has_book_text = books["parent_asin"].isin(book_level["parent_asin"])
has_documents = books["parent_asin"].isin(documents["parent_asin"])
book_coverage = pd.Series({
    "Features selected (with or without TOC)": int(book_level["source"].eq("features").sum()),
    "Description fallback (with or without TOC)": int(book_level["source"].eq("description").sum()),
    "TOC only": int((has_documents & ~has_book_text).sum()),
    "No documents": int((~has_documents).sum()),
}, name="Book records").to_frame()
assert book_coverage["Book records"].sum() == len(books)
book_coverage.loc["Total"] = len(books)
display(book_coverage)

display(Markdown(
    f"**Conclusion:** **{len(documents):,} documents** cover **{int(has_documents.sum()):,} "
    f"of {len(books):,} books**. Features supplies **{int(book_level['source'].eq('features').sum()):,}** "
    f"book-level rows and description supplies **{int(book_level['source'].eq('description').sum()):,}** "
    f"fallback rows. **{int((has_documents & ~has_book_text).sum()):,} books** use TOC text alone. "
    "Skipping blank TOC text affects this document table; the original rows remain in the TOC file."
))

,Book records
Features selected (with or without TOC),8991
Description fallback (with or without TOC),10
TOC only,33
No documents,0
Total,9034


**Conclusion:** **150,496 documents** cover **9,034 of 9,034 books**. Features supplies **8,991** book-level rows and description supplies **10** fallback rows. **33 books** use TOC text alone. Skipping blank TOC text affects this document table; the original rows remain in the TOC file.

<a id="section-4"></a>

## 4. Does the saved table match its inputs and the code?

**Objective:** Check every document, not just the selected example. Each document must have a unique ID, nonblank text, and a known book. Each book can have at most one book-level document.

We then rebuild the TOC and document tables in memory with `build_toc_features()` and `build_documents()`. `assert_frame_equal()` compares every row, column, value, data type, and row order with the saved tables. This catches stale or altered prepared output. The file checksums confirm that the saved inputs and output are unchanged. A failed check stops execution before the success conclusion.

In [6]:
assert documents["document_id"].notna().all() and documents["document_id"].is_unique
assert documents["parent_asin"].isin(books["parent_asin"]).all()
assert documents["text"].notna().all() and documents["text"].str.strip().ne("").all()
assert not documents.loc[documents["source"].ne("toc"), "parent_asin"].duplicated().any()

expected_toc = build_toc_features(books)
expected_toc["source_sha256"] = checksums_before[raw_path]
pd.testing.assert_frame_equal(toc_features, expected_toc, check_exact=True)

expected_documents = build_documents(books, expected_toc)
expected_documents["raw_source_sha256"] = checksums_before[raw_path]
expected_documents["toc_source_sha256"] = checksums_before[toc_path]
pd.testing.assert_frame_equal(documents, expected_documents, check_exact=True)
for path, checksum in checksums_before.items():
    assert file_checksum(path) == checksum, f"File changed during inspection: {path.name}"

display(Markdown(
    f"**Conclusion:** All **{len(documents):,} document rows and {len(documents.columns)} columns** "
    "match the current preparation code. Identifiers and source mappings pass, and all three files "
    "remain unchanged. This confirms reproducible preparation; it does not measure recommendation quality."
))

**Conclusion:** All **150,496 document rows and 11 columns** match the current preparation code. Identifiers and source mappings pass, and all three files remain unchanged. This confirms reproducible preparation; it does not measure recommendation quality.

**Next:** Use the document table to compare TOC-only retrieval with TOC plus the selected book-level text. Choose the embedding model and its tokenizer, split texts that exceed its input length, and keep the document and book IDs on each resulting passage. Define a small set of learning goals and relevance judgments before comparing ranking changes. The current table preserves complete text and contains no vectors.